# `ClassBase`

`ClassBase` is the common object foundation used by many Nematics3D classes. Users normally do **not** construct `ClassBase` directly. Instead, they encounter concrete objects—such as a disclination line, a field object, or a plotting object—that inherit a shared set of behaviors from it.

The purpose of this tutorial is therefore not to teach how to instantiate `ClassBase`. It is to teach how to **understand an unfamiliar Nematics3D object** once you have one.

## The mental model

A scientific class such as `DisclinationLine` has its own domain-specific job: it stores and analyzes a disclination line. `ClassBase` sits underneath that scientific behavior and supplies a common object protocol.

In practical terms, inheriting from `ClassBase` lets a Nematics3D object answer a few basic questions in a consistent way:

- **What kind of object am I?**
- **What data can you read from me?**
- **What does a particular attribute mean?**
- **What are you allowed to modify?**
- **What other Nematics3D objects am I connected to?**

This shared protocol is useful precisely because the concrete classes can otherwise be very different.

## Setup

The example below creates a small `DisclinationLine`. The details of disclination-line construction are not the subject of this tutorial; the object is only used as a concrete `ClassBase` descendant that we can inspect.

In [ ]:
import numpy as np

from nematics3d.classes.disclination_line import DisclinationLine, InputLine

defect_indices = np.array(
    [
        [0, 0, 0],
        [1, 0, 0],
        [2, 0, 0],
        [3, 0, 0],
    ],
    dtype=int,
)

line = DisclinationLine(
    InputLine(defect_indices=defect_indices),
    name="example line",
)
line

## First question: what is this object?

When you receive an unfamiliar Nematics3D object, start with:

```python
obj.show_doc()
```

`show_doc()` displays the class docstring of the **concrete class of that object**. It is therefore the quickest way to ask the object what it is designed to represent and what its main role is.

For our example:

In [ ]:
line.show_doc()

This is intentionally different from `show_readable_attrs()`. `show_doc()` answers **“what is this class?”**; `show_readable_attrs()` answers **“what does this particular object expose?”**.

## Second question: what can I read?

Use:

```python
obj.show_readable_attrs()
```

to list the registered user-readable attributes and their descriptions.

This is the main discovery tool for a `ClassBase` object. You do not need to know the class implementation before using it.

In [ ]:
line.show_readable_attrs()

If one attribute is unfamiliar, ask for only that description:

```python
obj.show_attr_doc("attribute_name")
```

For example:

In [ ]:
line.show_attr_doc("calc_defect_coords")

## Reading the attribute prefixes

Many Nematics3D classes deliberately use prefixes to tell you what role an attribute plays. These prefixes are part of the object vocabulary.

| Prefix or form | Meaning for a user |
| --- | --- |
| `raw_...` | Canonical stored input or core data. When a public alias exists, the `raw_` prefix can usually be omitted when reading. |
| `state_...` | Mutable runtime state: data describing the current state of the object rather than its original input. |
| `default_...` | A managed default-layer value used by the object. |
| `calc_...` | A computed result. It is normally read-only from the public interface. |
| `entity_...` | A computed or generated **object** rather than a scalar/array result. It is normally read-only. |
| `impl_...` | Internal implementation state. Ordinary users should normally ignore it. |
| no prefix, relation | A semantic link to another object, such as `owner` or `registry`. |
| no prefix, property | A normal Python property whose precise meaning is documented by the concrete class. |

The distinction is useful before you know any details of the class. For example, `raw_defect_indices` immediately tells you that the indices are core stored data, while `calc_defect_coords` tells you that the coordinates are a derived result.

### `raw_` attributes and public aliases

`raw_` has one additional convenience. If a class exposes `raw_xxx`, `ClassBase` normally also lets you read the same value as `xxx`.

For example:

In [ ]:
same_object = line.raw_defect_indices is line.defect_indices
same_values = np.array_equal(line.raw_defect_indices, line.defect_indices)

same_object, same_values

The explicit `raw_` name remains useful in documentation because it tells you the semantic role of the field. In ordinary analysis code, the shorter alias is often easier to read.

## What can I modify?

A `ClassBase` object is not an unrestricted Python attribute bag. Concrete classes can declare that some attributes are writable, computed, protected, or fixed after construction.

Instead of guessing, use:

```python
obj.show_modifiable_attrs()
```

In [ ]:
line.show_modifiable_attrs()

The result is **instance-aware**. A field can exist and be readable while not being modifiable on the current object.

This matters for scientific objects because changing core input after derived quantities have already been computed can make the object internally inconsistent. Some classes therefore freeze their core `raw_` or `state_` data and require you to construct a new object instead.

## Relations between objects

Nematics3D objects can also carry semantic links to other objects. `ClassBase` provides a common way to inspect those links.

Use:

```python
obj.show_relations()
```

to see currently bound relations and their targets.

In [ ]:
line.show_relations()

For objects embedded in a larger hierarchy, `show_relation_tree()` follows those links recursively:

```python
obj.show_relation_tree(depth=2)
```

This is most useful when working with registries, plotting objects, wrappers, or other object graphs where a single object is part of a larger structure.

## A practical inspection workflow

When an unfamiliar Nematics3D object appears in a notebook, a good default sequence is:

```python
obj.show_doc()                 # What is this object?
obj.show_readable_attrs()      # What can I read?
obj.show_attr_doc("...")       # What does this particular field mean?
obj.show_modifiable_attrs()    # What can I safely change?
obj.show_relations()           # What other objects is it connected to?
```

You will not always need every step. The point is that the same questions can be asked of many otherwise unrelated Nematics3D classes.

## What `ClassBase` is not

`ClassBase` does not perform the scientific calculation of its subclasses. It does not know how to smooth a line, diagonalize a Q tensor, build a figure, or analyze a defect.

A useful way to think about the inheritance is:

```text
ClassBase
    └── shared Nematics3D object behavior
            ├── naming and identity
            ├── attribute discovery
            ├── controlled assignment
            └── object relations

Concrete subclass
    └── the actual scientific or visualization behavior
```

So when you read documentation for `ClassBase`, you are learning the **common language of Nematics3D objects**, not a particular physical model or numerical algorithm.

# For developers

The user-facing ideas above are implemented through the class-level `__attr_defs__` schema and per-instance relation/assignment state. Developers defining a new `ClassBase` descendant should use the established attribute kinds and prefix conventions rather than creating a parallel naming system.

The important user-facing consequence is that a well-defined subclass automatically participates in the same discovery workflow: its class docstring is available through `show_doc()`, its registered attributes through `show_readable_attrs()`, its writable surface through `show_modifiable_attrs()`, and its declared object links through the relation helpers.

The internal schema machinery is intentionally not required knowledge for ordinary users and should be documented separately from the main usage path.